# Berlin AQI — Exploratory Data Analysis

Phase 1.1 of the planning doc. Goal: understand the data before writing any pipeline code.

**Seven questions to answer:**

1. How many Berlin stations are there, and what are their location IDs?
2. Which stations report PM2.5?
3. Which also report temperature and relative humidity?
4. How far back does each station's data go? (need ≥2 years)
5. How complete is the data per parameter per station?
6. What does the PM2.5 distribution look like? Class imbalance across risk categories?
7. Should SO₂, CO, and BC be included? (check variance)

We start with Berlin Mitte (`location_id=3019`) to validate the API shape, then expand.

## Setup

In [1]:
import os
import sys
from pathlib import Path

import httpx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Locate project root regardless of where jupyter was launched
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root (no pyproject.toml)")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

API_KEY = os.getenv("OPENAQ_API_KEY")
assert API_KEY, "OPENAQ_API_KEY not set in .env"

BASE = "https://api.openaq.org/v3"
HEADERS = {"X-API-Key": API_KEY}
# Berlin bounding box: minLon, minLat, maxLon, maxLat
BERLIN_BBOX = "13.088,52.338,13.761,52.675"
BERLIN_MITTE_ID = 3019

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 160)
print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: /Users/chandlershortlidge/berlin-aqi-MLOps


## Q1 — Berlin stations

Pull every location inside the Berlin bounding box. The planning doc expects ~21.

In [2]:
with httpx.Client(base_url=BASE, headers=HEADERS, timeout=30.0) as c:
    r = c.get("/locations", params={"bbox": BERLIN_BBOX, "limit": 1000})
    r.raise_for_status()
    locations = r.json()["results"]

print(f"Found {len(locations)} stations in Berlin bbox")

stations_df = pd.DataFrame(
    [
        {
            "id": l["id"],
            "name": l["name"],
            "provider": (l.get("provider") or {}).get("name"),
            "n_sensors": len(l.get("sensors", []) or []),
        }
        for l in locations
    ]
).sort_values("id")
stations_df

Found 33 stations in Berlin bbox


,id,name,provider,n_sensors
0,2993,Berlin Neukölln,EEA,6
1,3017,Berlin Karlshorst,EEA,3
2,3019,Berlin Mitte,EEA,4
3,3021,DEBE064,EEA Germany,3
4,3025,DEBE063,EEA Germany,4
5,3026,Berlin Schöneberg,EEA,2
6,3050,"Potsdam, Großbeerenstr.",EEA,4
7,3096,"Potsdam, Groß Glienicke",EEA,5
8,4582,Berlin Grunewald (3.5 m),EEA,5
9,4724,Blankenfelde-Mahlow,EEA,6


## Q2 + Q3 — Parameter coverage

Which stations report PM2.5, temperature, and relative humidity? Build a station × parameter coverage matrix.

In [3]:
sensor_rows = []
for l in locations:
    for s in l.get("sensors", []) or []:
        p = s.get("parameter") or {}
        sensor_rows.append(
            {
                "id": l["id"],
                "name": l["name"],
                "sensor_id": s["id"],
                "parameter": p.get("name"),
                "units": p.get("units"),
            }
        )

sensors_df = pd.DataFrame(sensor_rows)
print("Distinct parameters seen:", sorted(sensors_df["parameter"].dropna().unique()))

coverage = (
    sensors_df.assign(has=1)
    .pivot_table(index=["id", "name"], columns="parameter", values="has", fill_value=0)
    .astype(int)
)
coverage

Distinct parameters seen: ['co', 'no', 'no2', 'o3', 'pm1', 'pm10', 'pm25', 'relativehumidity', 'so2', 'temperature', 'um003']


,parameter,co,no,no2,o3,pm1,pm10,pm25,relativehumidity,so2,temperature,um003
id,name,,,,,,,,,,,
2993,Berlin Neukölln,1,1,1,1,0,1,1,0,0,0,0
3017,Berlin Karlshorst,0,1,1,0,0,0,0,0,1,0,0
3019,Berlin Mitte,0,1,1,0,0,1,1,0,0,0,0
3021,DEBE064,0,0,1,0,0,1,1,0,0,0,0
3025,DEBE063,0,1,1,0,0,1,1,0,0,0,0
3026,Berlin Schöneberg,0,1,1,0,0,0,0,0,0,0,0
3050,"Potsdam, Großbeerenstr.",0,1,1,0,0,1,1,0,0,0,0
3096,"Potsdam, Groß Glienicke",0,1,1,1,0,1,1,0,0,0,0
4582,Berlin Grunewald (3.5 m),0,1,1,1,0,1,1,0,0,0,0


In [4]:
# Export coverage matrix for reference (data/ is gitignored)
out_path = PROJECT_ROOT / "data" / "raw" / "parameter_coverage_berlin_districts.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
coverage.reset_index().to_csv(out_path, index=False)
print(f"Wrote {out_path}")

Wrote /Users/chandlershortlidge/berlin-aqi-MLOps/data/raw/parameter_coverage_berlin_districts.csv


## API shape check — Berlin Mitte (3019)

Before fetching 2 years of data, verify the response structure on a single station.

In [5]:
with httpx.Client(base_url=BASE, headers=HEADERS, timeout=30.0) as c:
    r = c.get(f"/locations/{BERLIN_MITTE_ID}")
    r.raise_for_status()
    mitte = r.json()["results"][0]

print("Name:    ", mitte["name"])
print("Country: ", (mitte.get("country") or {}).get("code"))
print("Timezone:", mitte.get("timezone"))
print("Provider:", (mitte.get("provider") or {}).get("name"))
print(f"Sensors ({len(mitte['sensors'])}):")
for s in mitte["sensors"]:
    p = s.get("parameter", {}) or {}
    first = (s.get("datetimeFirst") or {}).get("utc")
    last = (s.get("datetimeLast") or {}).get("utc")
    print(f"  - {p.get('name'):18s} ({p.get('units'):8s}) id={s['id']:<7} first={first} last={last}")

Name:     Berlin Mitte
Country:  DE
Timezone: Europe/Berlin
Provider: EEA
Sensors (4):
  - no                 (µg/m³   ) id=4275126 first=None last=None
  - no2                (µg/m³   ) id=6418    first=None last=None
  - pm10               (µg/m³   ) id=6422    first=None last=None
  - pm25               (µg/m³   ) id=1300119 first=None last=None


In [6]:
# Peek at a single hour record to see what fields we get back
pm25_sensor = next(
    s for s in mitte["sensors"] if (s.get("parameter") or {}).get("name") == "pm25"
)

with httpx.Client(base_url=BASE, headers=HEADERS, timeout=30.0) as c:
    r = c.get(f"/sensors/{pm25_sensor['id']}/hours", params={"limit": 1})
    r.raise_for_status()
    sample = r.json()["results"][0]

import json
print(json.dumps(sample, indent=2, default=str))

{
  "value": 15.1,
  "flagInfo": {
    "hasFlags": false
  },
  "parameter": {
    "id": 2,
    "name": "pm25",
    "units": "\u00b5g/m\u00b3",
    "displayName": null
  },
  "period": {
    "label": "1hour",
    "interval": "01:00:00",
    "datetimeFrom": {
      "utc": "2021-04-17T23:00:00Z",
      "local": "2021-04-18T01:00:00+02:00"
    },
    "datetimeTo": {
      "utc": "2021-04-18T00:00:00Z",
      "local": "2021-04-18T02:00:00+02:00"
    }
  },
  "coordinates": null,
  "summary": {
    "min": 15.13,
    "q02": 15.13,
    "q25": 15.13,
    "median": 15.13,
    "q75": 15.13,
    "q98": 15.13,
    "max": 15.13,
    "avg": 15.13,
    "sd": null
  },
  "coverage": {
    "expectedCount": 1,
    "expectedInterval": "01:00:00",
    "observedCount": 1,
    "observedInterval": "01:00:00",
    "percentComplete": 100.0,
    "percentCoverage": 100.0,
    "datetimeFrom": {
      "utc": "2021-04-17T23:00:00Z",
      "local": "2021-04-18T01:00:00+02:00"
    },
    "datetimeTo": {
      "utc": 

In [10]:
# Inspect what the bbox list endpoint actually returns for a sensor
with httpx.Client(base_url=BASE, headers=HEADERS, timeout=30.0) as c:
    r = c.get("/locations", params={"bbox": BERLIN_BBOX, "limit": 1000})
    r.raise_for_status()
    locs = r.json()["results"]

# Pick one PM2.5 sensor and show its full shape
for l in locs:
    for s in l.get("sensors", []) or []:
        if (s.get("parameter") or {}).get("name") == "pm25":
            import json
            print("keys on sensor object from /locations list:", sorted(s.keys()))
            print()
            print(json.dumps(s, indent=2, default=str))
            break
    else:
        continue
    break

keys on sensor object from /locations list: ['id', 'name', 'parameter']

{
  "id": 1300115,
  "name": "pm25 \u00b5g/m\u00b3",
  "parameter": {
    "id": 2,
    "name": "pm25",
    "units": "\u00b5g/m\u00b3",
    "displayName": "PM2.5"
  }
}


## Q4 — Time coverage per station

How many years of PM2.5 data does each station have? Need ≥2 years for training.

In [7]:
# The /locations bbox list endpoint returns lightweight sensor objects
# without datetimeFirst/Last. Fetch each PM2.5 sensor individually.
pm25_sensor_ids = sensors_df.loc[sensors_df["parameter"] == "pm25", "sensor_id"].tolist()
station_by_sensor = dict(
    sensors_df.loc[sensors_df["parameter"] == "pm25", ["sensor_id", "id"]].values
)
name_by_station = dict(stations_df[["id", "name"]].values)

time_rows = []
with httpx.Client(base_url=BASE, headers=HEADERS, timeout=30.0) as c:
    for sid in pm25_sensor_ids:
        r = c.get(f"/sensors/{sid}")
        r.raise_for_status()
        s = r.json()["results"][0]
        station_id = station_by_sensor[sid]
        time_rows.append(
            {
                "id": station_id,
                "name": name_by_station.get(station_id),
                "sensor_id": sid,
                "first": (s.get("datetimeFirst") or {}).get("utc"),
                "last": (s.get("datetimeLast") or {}).get("utc"),
            }
        )

time_df = pd.DataFrame(time_rows)
time_df["first"] = pd.to_datetime(time_df["first"], utc=True, errors="coerce")
time_df["last"] = pd.to_datetime(time_df["last"], utc=True, errors="coerce")
time_df["years"] = ((time_df["last"] - time_df["first"]).dt.days / 365.25).round(2)
time_df.sort_values("years", ascending=False)

,id,name,sensor_id,first,last,years
0,2993,Berlin Neukölln,1300115,NaT,NaT,NaN
1,3019,Berlin Mitte,1300119,NaT,NaT,NaN
2,3021,DEBE064,1300117,NaT,NaT,NaN
3,3025,DEBE063,1300118,NaT,NaT,NaN
4,3050,"Potsdam, Großbeerenstr.",5078552,NaT,NaT,NaN
5,3096,"Potsdam, Groß Glienicke",23604,NaT,NaT,NaN
6,4582,Berlin Grunewald (3.5 m),1300113,NaT,NaT,NaN
7,4724,Blankenfelde-Mahlow,11852,NaT,NaT,NaN
8,4761,Berlin Wedding,1300112,NaT,NaT,NaN
9,4762,Berlin Schildhornstraße,1300114,NaT,NaT,NaN


In [9]:
out_path = PROJECT_ROOT / "data" / "raw" / "berlin_districts_time.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
time_df.reset_index().to_csv(out_path, index=False)
print(f"Wrote {out_path}")

Wrote /Users/chandlershortlidge/berlin-aqi-MLOps/data/raw/berlin_districts_time.csv


## Q5 + Q6 — Deep dive on Berlin Mitte

Fetch 2 years of hourly data for station 3019 (via `src.ingest.fetch`, which already handles pagination).
Then answer:
- **Q5:** NaN % per parameter
- **Q6:** PM2.5 distribution and class balance across the custom AQI categories

In [8]:
from src.ingest import fetch

now = pd.Timestamp.now(tz="UTC")
iso = "%Y-%m-%dT%H:%M:%SZ"
datetime_from = (now - pd.Timedelta(days=730)).strftime(iso)
datetime_to = now.strftime(iso)

raw = fetch(BERLIN_MITTE_ID, datetime_from, datetime_to)
print(f"Rows: {len(raw):,}")
print(f"Date range: {raw['datetime'].min()} → {raw['datetime'].max()}")
raw.head()

HTTPStatusError: Client error '408 Request Timeout' for url 'https://api.openaq.org/v3/sensors/6422/hours?datetime_from=2024-04-20T13%3A53%3A01Z&datetime_to=2026-04-20T13%3A53%3A01Z&limit=1000&page=16'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/408

In [ ]:
# Q5 — NaN percentage per column
nan_pct = (raw.isna().mean() * 100).round(2).sort_values(ascending=False)
nan_pct.to_frame("nan_pct")

In [ ]:
# Q6a — PM2.5 distribution
fig, ax = plt.subplots(figsize=(10, 4))
raw["pm25"].dropna().clip(upper=150).hist(bins=60, ax=ax)
ax.set_xlabel("PM2.5 (µg/m³)  — clipped at 150 for readability")
ax.set_ylabel("Hours")
ax.set_title(f"PM2.5 hourly distribution — Berlin Mitte ({BERLIN_MITTE_ID})")
plt.tight_layout()
plt.show()

raw["pm25"].describe().round(2)

In [ ]:
# Q6b — Class balance across custom AQI categories
AQI_BINS = [0, 12, 35.4, 55.4, 150.4, 250.4, float("inf")]
AQI_LABELS = ["All Clear", "Low Risk", "Elevated", "Significant", "High", "Very High"]

aqi = pd.cut(raw["pm25"].dropna(), bins=AQI_BINS, labels=AQI_LABELS, include_lowest=True)
balance = aqi.value_counts().reindex(AQI_LABELS).fillna(0).astype(int)
balance_pct = (balance / balance.sum() * 100).round(2)

pd.DataFrame({"count": balance, "pct": balance_pct})

## Q7 — Variance of SO₂, CO, BC

A pollutant that barely changes in Berlin has no predictive value. Compute coefficient of variation (std / mean) for every pollutant available at Berlin Mitte.

In [ ]:
pollutants = [c for c in ["pm25", "pm10", "no2", "o3", "so2", "co"] if c in raw.columns]
stats = raw[pollutants].describe().T
stats["cv"] = (stats["std"] / stats["mean"]).round(3)
stats[["count", "mean", "std", "cv"]]

In [17]:
# Q7 follow-up: Mitte doesn't report SO2/CO/BC, so check which Berlin stations do
so2_co_bc_params = {"so2", "co", "bc"}
rows = sensors_df[sensors_df["parameter"].isin(so2_co_bc_params)]
print("Stations reporting SO2/CO/BC in the Berlin bbox:")
print(rows.groupby("parameter")["id"].nunique().to_string())
print()
print("Detail:")
rows[["id", "name", "parameter", "sensor_id"]].sort_values(["parameter", "id"])

Stations reporting SO2/CO/BC in the Berlin bbox:
parameter
co     4
so2    2

Detail:


,id,name,parameter,sensor_id
0,2993,Berlin Neukölln,co,11839788
36,4724,Blankenfelde-Mahlow,co,11856
47,4762,Berlin Schildhornstraße,co,11991
64,4767,Berlin Frankfurter Allee,co,12002
8,3017,Berlin Karlshorst,so2,6416
70,4767,Berlin Frankfurter Allee,so2,11999


## Summary — usable station list

A station is "usable" if it reports PM2.5 + temperature + relative humidity AND has ≥2 years of PM2.5 history.

In [ ]:
pm25_ids = set(sensors_df.loc[sensors_df["parameter"] == "pm25", "id"])
temp_ids = set(sensors_df.loc[sensors_df["parameter"] == "temperature", "id"])
rh_ids = set(sensors_df.loc[sensors_df["parameter"] == "relativehumidity", "id"])

print(f"Stations with PM2.5:              {len(pm25_ids)}")
print(f"Stations with temperature:        {len(temp_ids)}")
print(f"Stations with relative humidity:  {len(rh_ids)}")

has_all_three = pm25_ids & temp_ids & rh_ids
long_history = set(time_df.loc[time_df["years"] >= 2, "id"])
usable = sorted(has_all_three & long_history)

print(f"\nUsable stations (PM2.5 + temp + RH + ≥2y history): {len(usable)}")
print(usable)

stations_df[stations_df["id"].isin(usable)]

### Why zero usable stations

The summary returned **0** stations meeting all criteria. Diagnose where the gap is.

In [19]:
# Diagnose: where do PM2.5 stations and weather stations overlap (or not)?
pm25_ids = set(sensors_df.loc[sensors_df["parameter"] == "pm25", "id"])
temp_ids = set(sensors_df.loc[sensors_df["parameter"] == "temperature", "id"])
rh_ids = set(sensors_df.loc[sensors_df["parameter"] == "relativehumidity", "id"])

has_pm25_temp_rh = pm25_ids & temp_ids & rh_ids
has_pm25_and_weather = (pm25_ids & temp_ids) | (pm25_ids & rh_ids)

print(f"Stations with PM2.5 & temp & RH:   {len(has_pm25_temp_rh)}")
print(f"Stations with PM2.5 and temp:      {len(pm25_ids & temp_ids)}")
print(f"Stations with PM2.5 and RH:        {len(pm25_ids & rh_ids)}")
print(f"Stations with temp XOR PM2.5:      {len(temp_ids.symmetric_difference(pm25_ids))}")
print()
print("Which stations have temperature sensors?")
temp_station_names = stations_df[stations_df["id"].isin(temp_ids)][["id", "name", "provider"]]
print(temp_station_names.to_string(index=False))
print()
print("Do any of those have PM2.5?")
print(sorted(temp_ids & pm25_ids))

Stations with PM2.5 & temp & RH:   9
Stations with PM2.5 and temp:      9
Stations with PM2.5 and RH:        9
Stations with temp XOR PM2.5:      19

Which stations have temperature sensors?
     id                                name    provider
2977853          Berlin-Neukoelln Donaustr. AirGradient
3349325 Berlin-Neukoelln Donaustrasse North AirGradient
5130523      Berlin Spandau Paulsternstraße AirGradient
5932179            Schwerinstrasse Innenhof AirGradient
6121362                     Hohen Neuendorf AirGradient
6153720            Prinzregentenstr. Berlin AirGradient
6260071                  Potsdam-Waldstadt  AirGradient
6260438                             Balcony AirGradient
6270306                                  DM AirGradient

Do any of those have PM2.5?
[2977853, 3349325, 5130523, 5932179, 6121362, 6153720, 6260071, 6260438, 6270306]


## Weather supplement — Open-Meteo

Berlin government PM2.5 stations don't report temperature or relative humidity. Open-Meteo's free historical archive API covers Berlin hourly back to 1940 and fills the gap. `src.ingest.fetch_weather` and `fetch_coordinates` wrap it.

In [20]:
# Quick sanity check of the Open-Meteo archive API shape before writing code
import httpx
r = httpx.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": 52.5200,
        "longitude": 13.4050,
        "start_date": "2024-04-20",
        "end_date": "2024-04-22",
        "hourly": "temperature_2m,relative_humidity_2m",
        "timezone": "UTC",
    },
    timeout=30.0,
)
r.raise_for_status()
body = r.json()
print("Top-level keys:", list(body.keys()))
print("Hourly keys:   ", list(body["hourly"].keys()))
print(f"Returned {len(body['hourly']['time'])} hourly rows")
print("First 3 rows:")
for i in range(3):
    print(f"  {body['hourly']['time'][i]}  temp={body['hourly']['temperature_2m'][i]}°C  rh={body['hourly']['relative_humidity_2m'][i]}%")

Top-level keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
Hourly keys:    ['time', 'temperature_2m', 'relative_humidity_2m']
Returned 72 hourly rows
First 3 rows:
  2024-04-20T00:00  temp=5.4°C  rh=95%
  2024-04-20T01:00  temp=5.5°C  rh=95%
  2024-04-20T02:00  temp=5.2°C  rh=94%


In [21]:
import importlib
import src.ingest
importlib.reload(src.ingest)
from src.ingest import fetch_coordinates, fetch_weather

lat, lon = fetch_coordinates(BERLIN_MITTE_ID)
print(f"Berlin Mitte coords: {lat}, {lon}")

weather = fetch_weather(lat, lon, datetime_from, datetime_to)
print(f"Weather rows: {len(weather):,}")
print(f"Date range:   {weather['datetime'].min()} → {weather['datetime'].max()}")
print(f"Temp stats:   mean={weather['temperature'].mean():.1f}°C, min={weather['temperature'].min()}°C, max={weather['temperature'].max()}°C")
print(f"RH stats:     mean={weather['relative_humidity'].mean():.1f}%, min={weather['relative_humidity'].min()}%, max={weather['relative_humidity'].max()}%")

combined = raw.merge(weather, on="datetime", how="left")
print(f"\nCombined rows: {len(combined):,}")
print(f"Weather NaN after merge — temp: {combined['temperature'].isna().sum()}, RH: {combined['relative_humidity'].isna().sum()}")
combined[["datetime", "pm25", "no2", "temperature", "relative_humidity"]].head()

Berlin Mitte coords: 52.51360600001067, 13.41883300000022
Weather rows: 17,544
Date range:   2024-04-21 00:00:00+00:00 → 2026-04-21 23:00:00+00:00
Temp stats:   mean=10.7°C, min=-13.2°C, max=37.6°C
RH stats:     mean=74.5%, min=11%, max=100%

Combined rows: 15,248
Weather NaN after merge — temp: 0, RH: 0


,datetime,pm25,no2,temperature,relative_humidity
0,2024-04-21 06:00:00+00:00,8.60,6.64,1.8,73
1,2024-04-21 07:00:00+00:00,7.85,4.75,3.5,68
2,2024-04-21 08:00:00+00:00,6.91,4.49,5.0,62
3,2024-04-21 09:00:00+00:00,6.31,3.85,6.4,53
4,2024-04-21 10:00:00+00:00,5.90,1.52,6.6,53
